[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/09-polars.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/09-polars.ipynb)

# Polars: Fast DataFrames with Lazy Evaluation

**Module 4 — Data Science & Visualization** | Estimated time: 35 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Understand the core differences between Polars and Pandas
- Use `pl.DataFrame` and `pl.LazyFrame` (lazy evaluation with `scan_csv`)
- Apply the Polars Expression API: `filter`, `select`, `with_columns`, `group_by`, `agg`
- Use `.str`, `.dt`, and `.list` namespaces for typed column operations
- Inspect the lazy query plan with `.explain()`
- Benchmark Polars vs Pandas on 1 million row datasets
- Know when to choose Polars over Pandas

In [ ]:
!pip install polars --quiet

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

print(f'Polars  version: {pl.__version__}')
print(f'Pandas  version: {pd.__version__}')
print(f'NumPy   version: {np.__version__}')

rng = np.random.default_rng(42)

## 1. Polars vs Pandas — Syntax Comparison

Polars uses an **expression-based API**: instead of mutating a DataFrame in place, you compose expressions and execute them. This allows the query optimizer to reorder and parallelize operations automatically.

In [ ]:
# Side-by-side syntax comparison
data = {
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':   ['Engineering', 'Marketing', 'Engineering', 'HR', 'Marketing'],
    'salary': [95000, 72000, 110000, 68000, 85000],
    'years':  [5, 3, 8, 2, 6]
}

# Pandas
df_pd = pd.DataFrame(data)

# Polars
df_pl = pl.DataFrame(data)

print('=== Pandas DataFrame ===')
print(df_pd)
print(type(df_pd))

print('\n=== Polars DataFrame ===')
print(df_pl)
print(type(df_pl))

print('\n--- Selecting columns ---')
print('Pandas: df[["name","salary"]]')
print(df_pd[['name', 'salary']])
print('\nPolars: df.select(["name","salary"])')
print(df_pl.select(['name', 'salary']))

print('\n--- Filtering ---')
print('Pandas: df[df["salary"] > 80000]')
print(df_pd[df_pd['salary'] > 80000])
print('\nPolars: df.filter(pl.col("salary") > 80000)')
print(df_pl.filter(pl.col('salary') > 80000))

## 2. Creating Polars DataFrames and Type System

Polars uses Apache Arrow as its memory model, which means it has a strict, nullable type system and is zero-copy-compatible with other Arrow-based tools.

In [ ]:
# From dict
df = pl.DataFrame({
    'id':     [1, 2, 3, 4, 5],
    'name':   ['Alice', 'Bob', None, 'Dave', 'Eve'],
    'score':  [88.5, 92.0, 79.3, None, 85.1],
    'active': [True, False, True, True, False]
})
print('DataFrame:')
print(df)
print('\nSchema (name -> dtype):')
print(df.schema)
print('\nShape:', df.shape)
print('Null counts:', df.null_count())

# Key data types in Polars
print('\n--- Polars Core dtypes ---')
for dtype in [pl.Int32, pl.Int64, pl.Float32, pl.Float64,
              pl.Utf8, pl.Boolean, pl.Date, pl.Datetime]:
    print(f'  {dtype}')

# Type casting
df2 = df.with_columns([
    pl.col('id').cast(pl.Int32),
    pl.col('score').cast(pl.Float32)
])
print('\nAfter casting id->Int32, score->Float32:')
print(df2.dtypes)

## 3. The Expression API: filter, select, with_columns, group_by, agg

The expression API is the heart of Polars. Expressions are lazy descriptions of computations that Polars compiles into an optimized query plan.

In [ ]:
# Generate a richer dataset
np.random.seed(0)
n = 200
regions   = ['North', 'South', 'East', 'West']
products  = ['Widget', 'Gadget', 'Gizmo', 'Doohickey']

df_sales = pl.DataFrame({
    'order_id':  list(range(1, n + 1)),
    'region':    [regions[i % 4] for i in range(n)],
    'product':   [products[i % 4] for i in range(n)],
    'quantity':  rng.integers(1, 50, n).tolist(),
    'unit_price':rng.uniform(10, 200, n).round(2).tolist(),
    'discount':  rng.uniform(0, 0.3, n).round(3).tolist(),
    'date':      pl.date_range(
                    start=pl.date(2024, 1, 1),
                    end=pl.date(2024, 12, 31),
                    interval='2d',
                    eager=True
                )[:n].to_list()
})

print('Sales DataFrame:')
print(df_sales.head(5))
print('Shape:', df_sales.shape)

# with_columns: add derived columns
df_sales = df_sales.with_columns([
    (pl.col('quantity') * pl.col('unit_price') * (1 - pl.col('discount')))
        .round(2).alias('revenue'),
    pl.col('date').dt.month().alias('month'),
    pl.col('date').dt.quarter().alias('quarter')
])

print('\nWith derived columns:')
print(df_sales.head(3))

# filter
print('\nHigh-value orders (revenue > 500):')
print(df_sales.filter(pl.col('revenue') > 500).head(5))

# select with expressions
print('\nSelect with computed columns:')
print(df_sales.select([
    pl.col('product'),
    pl.col('revenue'),
    (pl.col('revenue') / pl.col('revenue').sum() * 100).round(2).alias('pct_total')
]).head(5))

# group_by + agg
print('\nGroupBy region:')
print(
    df_sales
    .group_by('region')
    .agg([
        pl.col('revenue').sum().alias('total_revenue'),
        pl.col('revenue').mean().round(2).alias('avg_revenue'),
        pl.col('order_id').count().alias('n_orders'),
        pl.col('discount').mean().round(3).alias('avg_discount')
    ])
    .sort('total_revenue', descending=True)
)

## 4. String, Date, and List Namespaces

In [ ]:
# String namespace
df_str = pl.DataFrame({'text': ['  Hello World  ', 'foo_bar_BAZ', 'Data123Science', None]})
print('String operations:')
print(df_str.with_columns([
    pl.col('text').str.strip_chars().alias('stripped'),
    pl.col('text').str.to_lowercase().alias('lower'),
    pl.col('text').str.contains('\\d').alias('has_digit'),
    pl.col('text').str.replace_all('_', ' ').alias('no_underscore')
]))

# Date/time namespace
print('\nDate operations:')
print(df_sales.select([
    pl.col('date'),
    pl.col('date').dt.year().alias('year'),
    pl.col('date').dt.month().alias('month'),
    pl.col('date').dt.weekday().alias('weekday'),  # 0=Mon, 6=Sun
    pl.col('date').dt.week().alias('week_of_year')
]).head(5))

# List namespace
df_list = pl.DataFrame({'nums': [[1, 2, 3], [4, 5], [7, 8, 9, 10]]})
print('\nList operations:')
print(df_list.with_columns([
    pl.col('nums').list.len().alias('length'),
    pl.col('nums').list.sum().alias('sum'),
    pl.col('nums').list.max().alias('max'),
    pl.col('nums').list.first().alias('first'),
]))

## 5. Lazy Evaluation with LazyFrame and scan_csv

A `LazyFrame` does NOT execute until you call `.collect()`. Polars builds a query graph and optimizes it (predicate pushdown, projection pushdown, parallel execution) before running anything.

In [ ]:
# Write a CSV so we can demonstrate scan_csv
csv_path = '/tmp/sales_data.csv'
df_sales.write_csv(csv_path)
print(f'CSV written to {csv_path}')

# scan_csv: lazy read
lazy_df = pl.scan_csv(csv_path)
print('Type:', type(lazy_df))
print('No data loaded yet — just a query plan.')

# Build a lazy query
lazy_query = (
    lazy_df
    .filter(pl.col('revenue') > 100)
    .filter(pl.col('quarter') == 1)
    .select(['region', 'product', 'revenue', 'month'])
    .group_by(['region', 'product'])
    .agg([
        pl.col('revenue').sum().alias('total_rev'),
        pl.col('revenue').count().alias('n_orders')
    ])
    .sort('total_rev', descending=True)
)

# Inspect the optimized plan BEFORE collecting
print('\n=== Optimized Query Plan (explain) ===')
print(lazy_query.explain())

# Execute and collect results
result = lazy_query.collect()
print('\n=== Query Result ===')
print(result)

## 6. Benchmark: Polars vs Pandas on 1 Million Rows

In [ ]:
N = 1_000_000
print(f'Generating {N:,} row dataset...')

# Generate data
regions_big  = rng.choice(['North','South','East','West'], N)
products_big = rng.choice(['Widget','Gadget','Gizmo','Doohickey'], N)
revenues_big = rng.exponential(200, N).round(2)
qtys_big     = rng.integers(1, 100, N)
discount_big = rng.uniform(0, 0.3, N).round(3)

# Write a large CSV
big_csv = '/tmp/big_sales.csv'
pd.DataFrame({
    'region':   regions_big,
    'product':  products_big,
    'revenue':  revenues_big,
    'quantity': qtys_big,
    'discount': discount_big
}).to_csv(big_csv, index=False)
print(f'CSV written ({N:,} rows)')

def time_it(label, func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    print(f'{label:<45} {elapsed*1000:>8.1f} ms')
    return elapsed

print(f'\n{"Operation":<45} {"Time":>8}')
print('-' * 55)

# --- Pandas benchmarks ---
df_pd_big = pd.read_csv(big_csv)
t_pd_read = time_it('Pandas: read_csv (1M rows)', lambda: pd.read_csv(big_csv))
t_pd_filter = time_it('Pandas: filter revenue > 100', lambda: df_pd_big[df_pd_big['revenue'] > 100])
t_pd_gb = time_it('Pandas: groupby + agg', lambda:
    df_pd_big.groupby(['region','product'])['revenue'].agg(['sum','mean','count']))

print()

# --- Polars benchmarks (eager) ---
df_pl_big = pl.read_csv(big_csv)
t_pl_read = time_it('Polars: read_csv (1M rows)', lambda: pl.read_csv(big_csv))
t_pl_filter = time_it('Polars: filter revenue > 100',
    lambda: df_pl_big.filter(pl.col('revenue') > 100))
t_pl_gb = time_it('Polars: group_by + agg', lambda:
    df_pl_big.group_by(['region','product']).agg([
        pl.col('revenue').sum(),
        pl.col('revenue').mean(),
        pl.col('revenue').count()
    ]))

print()

# --- Polars lazy benchmarks ---
t_pl_lazy = time_it('Polars: lazy scan + filter + groupby + collect',
    lambda: (
        pl.scan_csv(big_csv)
        .filter(pl.col('revenue') > 100)
        .group_by(['region','product'])
        .agg([pl.col('revenue').sum(), pl.col('revenue').mean()])
        .collect()
    )
)

print('\n--- Speedup Summary ---')
for op, t_pd, t_pl in [
    ('read_csv',  t_pd_read,   t_pl_read),
    ('filter',    t_pd_filter, t_pl_filter),
    ('group_by',  t_pd_gb,     t_pl_gb),
]:
    speedup = t_pd / t_pl
    print(f'  {op:<12}: Polars {speedup:.1f}x faster')

## 7. Benchmark Visualization

In [ ]:
operations = ['read_csv', 'filter', 'groupby']
pandas_times = [t_pd_read*1000, t_pd_filter*1000, t_pd_gb*1000]
polars_times = [t_pl_read*1000, t_pl_filter*1000, t_pl_gb*1000]

x = np.arange(len(operations))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, pandas_times, width, label='Pandas',  color='#1f77b4', alpha=0.85)
bars2 = ax.bar(x + width/2, polars_times, width, label='Polars',  color='#ff7f0e', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{bar.get_height():.0f}ms', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{bar.get_height():.0f}ms', ha='center', va='bottom', fontsize=9)

ax.set_title('Pandas vs Polars — 1 Million Row Benchmark', fontsize=13)
ax.set_ylabel('Time (milliseconds, lower is better)')
ax.set_xticks(x)
ax.set_xticklabels(operations)
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. When to Use Polars vs Pandas

A practical decision guide based on your use case.

In [ ]:
decision_guide = {
    'Scenario': [
        'Dataset < 100K rows',
        'Dataset 100K – 5M rows',
        'Dataset > 5M rows / large files',
        'Interactive data exploration',
        'Sklearn / ML pipelines',
        'SQL-style transformations',
        'Streaming / out-of-core data',
        'Ecosystem compatibility (Excel, statsmodels, etc.)',
        'Speed is critical (ETL, data engineering)',
        'Learning / teaching data science',
    ],
    'Recommended': [
        'Either (Pandas easier)',
        'Either (Polars faster)',
        'Polars (lazy + streaming)',
        'Pandas (Jupyter display, plot integration)',
        'Pandas (sklearn expects pandas/numpy)',
        'Polars (expression API is closer to SQL)',
        'Polars (LazyFrame + streaming)',
        'Pandas (broader ecosystem)',
        'Polars (multi-threaded, Rust backend)',
        'Pandas (more tutorials, wider adoption)',
    ],
    'Why': [
        'Overhead difference is negligible',
        'Polars is 2-10x faster; both work fine',
        'Lazy evaluation avoids OOM errors',
        'Pandas df.head(), df.plot() etc.',
        '.to_numpy() works on both but Pandas is standard',
        'group_by/filter/join syntax maps naturally',
        'scan_csv + .collect() processes chunk-by-chunk',
        'Pandas integrates with openpyxl, statsmodels, etc.',
        'Rust backend, no GIL, parallel execution',
        'StackOverflow, books, courses use Pandas'
    ]
}

df_guide = pl.DataFrame(decision_guide)
print('When to Use Polars vs Pandas:')
print(df_guide.to_pandas().to_string(index=False))

print('\n--- Key Polars Advantages ---')
advantages = [
    'Multi-threaded execution (uses all CPU cores)',
    'Lazy query optimization (predicate/projection pushdown)',
    'Out-of-core / streaming for datasets larger than RAM',
    'Consistent, expression-based API (no index ambiguity)',
    'Apache Arrow memory format (zero-copy interop)',
    'Better null handling (always nullable, no NaN vs None confusion)',
    'Type safety: schema enforced at compile time',
]
for adv in advantages:
    print(f'  + {adv}')

print('\n--- Key Pandas Advantages ---')
advantages_pd = [
    'Larger ecosystem (sklearn, statsmodels, plotly, etc.)',
    'More tutorials, books, and community resources',
    'Better interactive display in Jupyter',
    'MultiIndex for hierarchical data',
    'More date/time offset types (BusinessDay, etc.)',
]
for adv in advantages_pd:
    print(f'  + {adv}')

## Practice Exercises

**Exercise 1 — Polars Expression Chaining**
Using the `df_sales` Polars DataFrame, write a single chained expression (no intermediate variables) that: filters orders where `discount > 0.15`, adds a column `net_revenue = revenue * (1 - discount)`, groups by `quarter` and `region`, and aggregates the total `net_revenue` and average `quantity`. Sort by `net_revenue` descending.

**Exercise 2 — Lazy Pipeline with explain()**
Build a lazy pipeline on the big CSV that: scans the file lazily, filters `quantity > 50`, selects only `region`, `product`, `revenue`, groups by `region` and `product`, sums `revenue`. Call `.explain()` and identify which optimization (predicate pushdown, projection pushdown) Polars applies. Then `.collect()` and display the result.

**Exercise 3 — Polars String and Date Operations**
Create a Polars DataFrame with columns: `full_name` (e.g. 'alice smith'), `signup_date` (strings like '2024-03-15'), `tags` (list of strings like ['python', 'ml']). Use the `.str`, `.dt`, and `.list` namespaces to: title-case the names, parse the date strings to Date type and extract the month, count how many tags each person has, and filter to people who have more than 1 tag.